In [ ]:
from pathlib import Path
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from matplotlib.colors import to_rgba

# =========================================================
# CONFIGURAÇÃO
# =========================================================
OUTPUT_DIR = Path(".")   # troque para Path("/alguma/pasta") se quiser
MULTIPLE_K = 3           # categoria "Multiples of k"
FPS_MP4 = 10
FPS_GIF = 8
DPI = 100

MP4_NAME = "numbers_reintroduced_9x16.mp4"
GIF_NAME = "numbers_reintroduced_9x16.gif"
PREVIEW_NAME = "numbers_reintroduced_preview.png"

# =========================================================
# FUNÇÕES AUXILIARES DE TEORIA DOS NÚMEROS
# =========================================================
def is_prime(n):
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    r = int(n**0.5)
    for d in range(3, r + 1, 2):
        if n % d == 0:
            return False
    return True

def prime_factor_count_with_multiplicity(n):
    x = n
    count = 0
    d = 2
    while d * d <= x:
        while x % d == 0:
            count += 1
            x //= d
        d += 1 if d == 2 else 2
    if x > 1:
        count += 1
    return count

def proper_divisor_sum(n):
    if n == 1:
        return 0
    s = 1
    r = int(n**0.5)
    for d in range(2, r + 1):
        if n % d == 0:
            q = n // d
            s += d
            if q != d:
                s += q
    return s

def is_happy(n):
    seen = set()
    x = n
    while x != 1 and x not in seen:
        seen.add(x)
        x = sum(int(c) ** 2 for c in str(x))
    return x == 1

def is_kaprekar(n):
    # Definição decimal padrão de Kaprekar
    if n == 1:
        return True
    sq = n * n
    d = 10 ** len(str(n))
    left, right = sq // d, sq % d
    return right > 0 and left + right == n

def triangular_numbers(limit):
    vals = []
    k = 1
    while True:
        t = k * (k + 1) // 2
        if t > limit:
            break
        vals.append(t)
        k += 1
    return vals

def pentagonal_numbers(limit):
    vals = []
    k = 1
    while True:
        p = k * (3 * k - 1) // 2
        if p > limit:
            break
        vals.append(p)
        k += 1
    return vals

def pell_numbers(limit):
    vals = []
    a, b = 0, 1
    while b <= limit:
        vals.append(b)
        a, b = b, 2 * b + a
    return vals

# =========================================================
# DADOS DAS CATEGORIAS
# =========================================================
N = list(range(1, 101))

categories = [
    ("Natural Numbers", set(N), "#ff3b30"),
    ("Even Numbers", {n for n in N if n % 2 == 0}, "#ff9f0a"),
    (f"Multiples of {MULTIPLE_K}", {n for n in N if n % MULTIPLE_K == 0}, "#ffb300"),
    ("Prime Numbers", {n for n in N if is_prime(n)}, "#ffd60a"),
    ("Semiprime Numbers", {n for n in N if n >= 4 and prime_factor_count_with_multiplicity(n) == 2}, "#f4e285"),
    ("Cube Numbers", {1, 8, 27, 64}, "#34c759"),
    ("Triangular Numbers", set(triangular_numbers(100)), "#32d74b"),
    ("Pentagonal Numbers", set(pentagonal_numbers(100)), "#30d158"),
    ("Perfect Numbers", {n for n in N if n > 1 and proper_divisor_sum(n) == n}, "#64d2ff"),
    ("Deficient Numbers", {n for n in N if proper_divisor_sum(n) < n}, "#66d9e8"),
    ("Decimal Harshad Numbers", {n for n in N if n % sum(int(c) for c in str(n)) == 0}, "#5ac8fa"),
    ("Binary Palindromic Numbers", {n for n in N if bin(n)[2:] == bin(n)[2:][::-1]}, "#0a84ff"),
    ("Happy Numbers", {n for n in N if is_happy(n)}, "#bf5af2"),
    ("Evil Numbers", {n for n in N if bin(n).count("1") % 2 == 0}, "#ff2d55"),
    ("Kaprekar Numbers", {n for n in N if is_kaprekar(n)}, "#ff375f"),
    ("Pell Numbers", set(pell_numbers(100)), "#ff453a"),
]

# =========================================================
# PARÂMETROS DA ANIMAÇÃO
# =========================================================
# Cada cena terá:
# - reveal_frames: acendimento progressivo dos elementos
# - hold_frames: manutenção da cena final
reveal_frames = 3
hold_frames = 4
frames_per_scene = reveal_frames + hold_frames
total_frames = frames_per_scene * len(categories)

# =========================================================
# CONSTRUÇÃO DA FIGURA
# =========================================================
fig, ax = plt.subplots(figsize=(3.6, 6.4), dpi=DPI)   # 360x640 px -> 9:16
fig.patch.set_facecolor("black")
ax.set_facecolor("black")
ax.set_xlim(-0.6, 9.6)
ax.set_ylim(-0.7, 11.9)
ax.axis("off")

title = ax.text(
    4.5, 11.1, "",
    ha="center", va="center",
    fontsize=16, fontweight="bold", color="white"
)

subtitle = ax.text(
    4.5, 10.55, "",
    ha="center", va="center",
    fontsize=8.5, color="#bcbcbc"
)

footer = ax.text(
    4.5, -0.35, "Numbers 1 to 100",
    ha="center", va="center",
    fontsize=8.5, color="#888888"
)

# Grelha 10x10 de textos
text_objects = {}
for n in N:
    row = (n - 1) // 10
    col = (n - 1) % 10
    x = col
    y = 9 - row

    txt = ax.text(
        x, y, str(n),
        ha="center", va="center",
        fontsize=9.8, fontweight="bold", color="#666666",
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="#0d0d0d",
            edgecolor="#232323",
            linewidth=0.8
        )
    )
    text_objects[n] = txt

# =========================================================
# FUNÇÕES DE ANIMAÇÃO
# =========================================================
def init():
    return [title, subtitle, footer] + list(text_objects.values())

def update(frame):
    scene_idx = min(frame // frames_per_scene, len(categories) - 1)
    local = frame % frames_per_scene

    scene_title, scene_values, scene_color = categories[scene_idx]
    sorted_vals = sorted(scene_values)

    # Acendimento progressivo
    if local < reveal_frames:
        k = max(1, math.ceil(len(sorted_vals) * (local + 1) / reveal_frames))
        active = set(sorted_vals[:k])
    else:
        active = scene_values

    title.set_text(scene_title)
    title.set_color(scene_color)
    subtitle.set_text(f"{len(scene_values)} highlighted")

    for n, txt in text_objects.items():
        bb = txt.get_bbox_patch()
        if n in active:
            txt.set_color("white")
            bb.set_facecolor(to_rgba(scene_color, 0.28))
            bb.set_edgecolor(scene_color)
            bb.set_linewidth(1.1)
        else:
            txt.set_color("#6a6a6a")
            bb.set_facecolor("#0d0d0d")
            bb.set_edgecolor("#232323")
            bb.set_linewidth(0.8)

    return [title, subtitle, footer] + list(text_objects.values())

# =========================================================
# CRIAÇÃO DA ANIMAÇÃO
# =========================================================
anim = FuncAnimation(
    fig,
    update,
    init_func=init,
    frames=total_frames,
    interval=1000 / FPS_MP4,
    blit=False
)

# =========================================================
# EXPORTAÇÃO
# =========================================================
preview_path = OUTPUT_DIR / PREVIEW_NAME
mp4_path = OUTPUT_DIR / MP4_NAME
gif_path = OUTPUT_DIR / GIF_NAME

# Salva uma imagem de preview
update(0)
fig.savefig(preview_path, facecolor=fig.get_facecolor(), bbox_inches="tight")

# Salva MP4
try:
    mp4_writer = FFMpegWriter(fps=FPS_MP4, bitrate=1800)
    anim.save(mp4_path, writer=mp4_writer, dpi=DPI)
    print(f"MP4 salvo em: {mp4_path.resolve()}")
except Exception as e:
    print("Erro ao salvar MP4.")
    print("Verifique se o ffmpeg está instalado.")
    print("Detalhes:", e)

# Salva GIF
try:
    gif_writer = PillowWriter(fps=FPS_GIF)
    anim.save(gif_path, writer=gif_writer, dpi=DPI)
    print(f"GIF salvo em: {gif_path.resolve()}")
except Exception as e:
    print("Erro ao salvar GIF.")
    print("Verifique se Pillow está instalado.")
    print("Detalhes:", e)

plt.close(fig)

print(f"Preview salvo em: {preview_path.resolve()}")